# Data Preprocessing

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("NYC Airport Taxi Preprocessing")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "code"
    else CURRENT_DIR
)

DATA_DIR = PROJECT_ROOT / "data"
TLC_DIR = DATA_DIR / "tlc_data"
ZONE_DIR = DATA_DIR / "taxi_zones"
WEATHER_DIR = DATA_DIR / "weather"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

START_DATETIME = "2024-01-01 00:00:00"
END_DATETIME = "2024-07-01 00:00:00"

AIRPORT_ZONES_IDS = {
    132: "JFK Airport",
    138: "LaGuardia Airport"
}

taxi_files = sorted(TLC_DIR.glob("yellow_tripdata_2024-*.parquet"))

assert len(taxi_files) == 6
print(f"Expected 6 taxi trip data files, found {len(taxi_files)}")

## 1. Load Taxi Data

In [ ]:
taxi_raw = (
    spark.read.parquet(*[str(file) for file in taxi_files])
    .withColumn(
        "source_month",
        F.regexp_extract(
            F.input_file_name(),
            r"yellow_tripdata_(\d{4}-\d{2})\.parquet",
            1
        )
    )
)

In [ ]:
monthly_raw_summary = (
    taxi_raw
    .groupBy("source_month")
    .agg(
        F.count("*").alias("trip_count"),
        F.min("tpep_pickup_datetime").alias("minimum_pickup_datetime"),
        F.max("tpep_pickup_datetime").alias("maximum_pickup_datetime")
    )
    .orderBy("source_month")
)

monthly_raw_summary.show(truncate=False)

raw_trip_count = taxi_raw.count()

print(f"\nTotal raw trips: {raw_trip_count:,}")

In [ ]:
period_condition = (
    (F.col("tpep_pickup_datetime") >= START_DATETIME) &
    (F.col("tpep_pickup_datetime") < END_DATETIME)
)

taxi_in_period = taxi_raw.filter(period_condition)

in_period_trip_count = taxi_in_period.count()
outside_period_count = raw_trip_count - in_period_trip_count

print(f"Raw trips: {raw_trip_count:,}")
print(f"Trips in analysis period: {in_period_trip_count:,}")
print(f"Trips removed: {outside_period_count:,}")

## 2. Select Airport Trips

In [ ]:
airport_trips_raw = taxi_in_period.filter(
    F.col("DOLocationID").isin(list(AIRPORT_ZONES_IDS.keys()))
)

airport_count_rows = (
    airport_trips_raw
    .groupBy("DOLocationID")
    .agg(F.count("*").alias("trip_count"))
    .orderBy("DOLocationID")
    .collect()
)

airport_trip_count = sum(row["trip_count"] for row in airport_count_rows)

for row in airport_count_rows:
    print(f"Zone {row['DOLocationID']} "
          f"{row['trip_count']:,} trips")

print(f"Total airport trips: {airport_trip_count:,}")

In [ ]:
zone_lookup = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ZONE_DIR / "taxi_zone_lookup.csv"))
)

airport_zones = (
    zone_lookup
    .filter(F.col("LocationID").isin(132, 138))
    .select("LocationID", "Borough", "Zone", "service_zone")
    .orderBy("LocationID")
)

airport_zones.show(truncate=False)

In [ ]:
airport_lookup = airport_zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Zone").alias("airport")
)

airport_trips = airport_trips_raw.join(
    F.broadcast(airport_lookup),
    on="DOLocationID",
    how="inner"
)

## 3. Clean Taxi Trips

In [ ]:
missing_counts = (
    airport_trips
    .select(
        [
            F.sum(
                F.col(column).isNull().cast("int")
            ).alias(column)
            for column in airport_trips.columns
        ]
    )
    .first()
    .asDict()
)

columns_with_missing_values = {
    column: count
    for column, count in missing_counts.items()
    if count > 0
}

print(f"Columns assessed: {len(missing_counts)}")

if columns_with_missing_values:
    for column, count in columns_with_missing_values.items():
        percentage = count / airport_trip_count * 100

        print(
            f"{column}: {count:,} "
            f"({percentage:.2f}%)"
        )
else:
    print("No missing values found.")

In [ ]:
airport_trips = airport_trips.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime") -
        F.unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

airport_trips.select(
    "trip_duration_minutes",
    "trip_distance"
).summary(
    "count",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show()

In [ ]:
outlier_assessment = airport_trips.agg(
    F.expr(
        """
        percentile_approx(
            trip_duration_minutes,
            array(0.001, 0.01, 0.99, 0.999)
        )
        """
    ).alias("duration_percentiles"),
    
    F.expr(
        """
        percentile_approx(
            trip_distance,
            array(0.001, 0.01, 0.99, 0.999)
        )
        """
    ).alias("distance_percentiles"),
    
    F.sum(
        (F.col("trip_duration_minutes") <= 0).cast("int")
    ).alias("non_positive_duration"),
    
    F.sum(
        (F.col("trip_duration_minutes") > 180).cast("int")
    ).alias("duration_over_180"),
    
    F.sum(
        (F.col("trip_distance") <= 0).cast("int")
    ).alias("non_positive_distance"),
    
    F.sum(
        (F.col("trip_distance") > 100).cast("int")
    ).alias("distance_over_100")
)

outlier_assessment.show(truncate=False)

In [ ]:
airport_trips.agg(
    F.sum(
        (F.col("trip_duration_minutes") < 1).cast("int")
    ).alias("duration_under_1_minute"),

    F.sum(
        ((F.col("trip_distance") <= 0) & (F.col("trip_duration_minutes") < 1)).cast("int")
    ).alias("zero_distance_and_short_duration"),

    F.sum(
        ((F.col("trip_distance") <= 0) & 
         (F.col("trip_duration_minutes") >= 1) &
         (F.col("trip_duration_minutes") <= 180) &
         (F.col("PULocationID") != F.col("DOLocationID"))
        ).cast("int")
    ).alias("zero_distance_but_usable_time_and_zones"),

    F.sum((F.col("PULocationID") == F.col("DOLocationID")).cast("int")
          ).alias("same_pickup_dropoff_zone")
).show(truncate=False)

In [ ]:
valid_trip_condition = (
    (F.col("trip_duration_minutes") >= 1) &
    (F.col("trip_duration_minutes") <= 180) &
    (F.col("PULocationID") != F.col("DOLocationID")) &
    (F.col("trip_distance") <= 100)
)

clean_airport_trips = airport_trips.filter(valid_trip_condition)

clean_trip_count = clean_airport_trips.count()
removed_trip_count = airport_trip_count - clean_trip_count

print(f"Airport trips before cleaning: {airport_trip_count:,}")
print(f"Airport trips after cleaning: {clean_trip_count:,}")
print(
    f"Trips removed: {removed_trip_count:,} "
    f"({removed_trip_count / airport_trip_count * 100:.2f}%)"
)

In [ ]:
deduplicated_trips = clean_airport_trips.dropDuplicates()

deduplicated_count = deduplicated_trips.count()
duplicate_count = clean_trip_count - deduplicated_count

print(f"Exact duplicate trips: {duplicate_count:,}")

In [ ]:
clean_airport_trips = (
    clean_airport_trips
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour_timestamp", F.date_trunc("hour", "tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_day_name", F.date_format("tpep_pickup_datetime", "E"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("is_weekend", F.dayofweek("tpep_pickup_datetime").isin(1,7).cast("int"))
)

In [ ]:
pickup_lookup = zone_lookup.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
    F.col("service_zone").alias("pickup_service_zone")
)

clean_airport_trips = clean_airport_trips.join(
    F.broadcast(pickup_lookup),
    on="PULocationID",
    how="left"
)

missing_pickup_zones = (
    clean_airport_trips
    .filter(F.col("pickup_zone").isNull())
    .count()
)

print(f"Trips without a matched pickup zone: {missing_pickup_zones:,}")

## 4. Add Weather Data

In [ ]:
jfk_weather_raw = (
    spark.read
    .option("header", True)
    .csv(str(WEATHER_DIR / "jfk_weather_2024.csv"))
)

lga_weather_raw = (
    spark.read
    .option("header", True)
    .csv(str(WEATHER_DIR / "lga_weather_2024.csv"))
)

In [ ]:
def numeric_weather_value(column_name):
    extracted_value = F.regexp_extract(
        F.col(column_name).cast("string"),
        r"-?\d+(?:\.\d+)?",
        0
    )

    return F.when(
        extracted_value != "",
        extracted_value.cast("double")
    ).otherwise(
        F.lit(None).cast("double")
    )

def prepare_weather(weather_df, airport_name):
    return weather_df.select(
        F.lit(airport_name).alias("airport"),
        F.col("DATE").cast("timestamp").alias("weather_timestamp"),

        numeric_weather_value(
            "HourlyDryBulbTemperature"
        ).alias("temperature_f"),

        F.when(F.upper(F.trim(F.col("HourlyPrecipitation").cast("string"))
                       )== "T",F.lit(0.0)
        ).otherwise(
            numeric_weather_value("HourlyPrecipitation")
        ).alias("precipitation_inches"),

        numeric_weather_value("HourlyVisibility").alias("visibility_miles"),
        numeric_weather_value("HourlyWindSpeed").alias("wind_speed_mph"),
        numeric_weather_value("HourlyRelativeHumidity").alias("relative_humidity"),

        F.col("HourlyPresentWeatherType").alias("present_weather")
    )

weather_selected = (
    prepare_weather(jfk_weather_raw, "JFK Airport")
    .unionByName(prepare_weather(lga_weather_raw, "LaGuardia Airport"))
    .filter(
        (F.col("weather_timestamp") >= START_DATETIME) &
        (F.col("weather_timestamp") < END_DATETIME)
    )
)

In [ ]:
weather_hourly = (
    weather_selected
    .withColumn(
        "weather_hour", 
        F.date_trunc(
            "hour",
            F.col("weather_timestamp") +
            F.expr("INTERVAL 30 MINUTES")
            )
    )
    .groupBy("airport", "weather_hour")
    .agg(
        F.avg("temperature_f").alias("temperature_f"),
        F.max("precipitation_inches").alias("precipitation_inches"),
        F.min("visibility_miles").alias("visibility_miles"),
        F.avg("wind_speed_mph").alias("wind_speed_mph"),
        F.avg("relative_humidity").alias("relative_humidity"),
        F.first("present_weather", True).alias("present_weather")
    )
)

In [ ]:
weather_hourly = weather_hourly.filter(
    (F.col("weather_hour") >= START_DATETIME) &
    (F.col("weather_hour") < END_DATETIME)
)

weather_quality_summary = (
    weather_hourly
    .groupBy("airport")
    .agg(
        F.count("*").alias("available_hours"),
        F.min("weather_hour").alias("first_hour"),
        F.max("weather_hour").alias("last_hour"),

        F.sum(
            F.col("temperature_f").isNull().cast("int")
            ).alias("missing_temperature"),

        F.sum(
            F.col("precipitation_inches").isNull().cast("int")
        ).alias("missing_precipitation"),

        F.sum(
            F.col("visibility_miles").isNull().cast("int")
        ).alias("missing_visibility"),

        F.sum(
            F.col("wind_speed_mph").isNull().cast("int")
        ).alias("missing_wind_speed"),

        F.sum(
            F.col("relative_humidity").isNull().cast("int")
        ).alias("missing_relative_humidity")
    ).orderBy("airport")
)

weather_quality_summary.show(truncate=False)

In [ ]:
missing_precipitation_check = (
    weather_hourly
    .filter(F.col("precipitation_inches").isNull())
    .groupBy("airport")
    .agg(
        F.count("*").alias("missing_precipitation_hours"),

        F.sum(
            F.col("present_weather").isNotNull().cast("int")
        ).alias("hours_with_weather_code"),

        F.sum(
            (F.col("visibility_miles") < 5).cast("int")
        ).alias("hours_with_reduced_visibility_under_5"),

        F.min("visibility_miles").alias("minimum_visibility")
    ).orderBy("airport")
)

missing_precipitation_check.show(truncate=False)

In [ ]:
weather_hourly = (
    weather_hourly
    .withColumn("precipitation_missing",F.col("precipitation_inches")
        .isNull()
        .cast("int")
    )
    .withColumn(
        "precipitation_for_model",F.coalesce(F.col("precipitation_inches"),F.lit(0.0))
        )
    .withColumn("weather_code",F.upper(F.coalesce(F.col("present_weather"), F.lit(""))))
    .withColumn(
        "precipitation_event",
        ((F.col("precipitation_for_model") > 0) |
            F.col("weather_code").rlike(
                "RA|DZ|SN|SG|PL")
        ).cast("int")
    )
    .withColumn("snow_event", F.col("weather_code").rlike("SN|SG|PL").cast("int"))
    .withColumn("fog_or_low_visibility", 
                (F.col("weather_code").rlike("FG|BR")|
                 (F.col("visibility_miles") < 5)
                ).cast("int"))
)

In [ ]:
weather_for_join = weather_hourly.select(
    "airport",

    F.col("weather_hour").cast("timestamp_ntz")
    .alias("pickup_hour_timestamp"),

    "temperature_f",
    "precipitation_inches",
    "precipitation_missing",
    "precipitation_for_model",
    "visibility_miles",
    "wind_speed_mph",
    "relative_humidity",
    "present_weather",
    "precipitation_event",
    "snow_event",
    "fog_or_low_visibility"
).withColumn(
    "weather_matched",
    F.lit(1)
)

airport_weather_trips = clean_airport_trips.join(
    F.broadcast(weather_for_join),
    on=["airport", "pickup_hour_timestamp"],
    how="left"
)

join_summary = airport_weather_trips.agg(
    F.count("*").alias("total_trips"),

    F.sum(
        F.col("weather_matched").isNotNull().cast("int")
    ).alias("weather_matched_trips"),

    F.sum(
        F.col("weather_matched").isNull().cast("int")
    ).alias("weather_unmatched_trips")
).first()

matched_count = join_summary["weather_matched_trips"]
unmatched_count = join_summary["weather_unmatched_trips"]

print(f"Total cleaned trips: {join_summary['total_trips']:,}")
print(f"Trips matched to weather: {matched_count:,}")
print(f"Trips without weather match: {unmatched_count:,}")
print("Weather match rate:")
print(f"{matched_count / join_summary['total_trips'] * 100:.2f}%")

In [ ]:
complete_weather_condition = (
    F.col("weather_matched").isNotNull() &
    F.col("temperature_f").isNotNull() &
    F.col("visibility_miles").isNotNull() &
    F.col("wind_speed_mph").isNotNull() &
    F.col("relative_humidity").isNotNull()
)

analysis_data = airport_weather_trips.filter(complete_weather_condition)

analysis_trip_count = analysis_data.count()
weather_removed_count = clean_trip_count - analysis_trip_count

print(f"Trips before weather matching: {clean_trip_count:,}")
print(f"Final analysis trips: {analysis_trip_count:,}")
print("Trips removed for incomplete weather data: ")
print(f"{weather_removed_count:,}")
print(f"({weather_removed_count / clean_trip_count * 100:.2f}%)")

In [ ]:
final_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "pickup_borough",
    "pickup_zone",
    "pickup_service_zone",
    "airport",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_date",
    "pickup_hour_timestamp",
    "pickup_month",
    "pickup_day_of_week",
    "pickup_day_name",
    "pickup_hour",
    "is_weekend",
    "temperature_f",
    "precipitation_inches",
    "precipitation_missing",
    "precipitation_for_model",
    "visibility_miles",
    "wind_speed_mph",
    "relative_humidity",
    "present_weather",
    "precipitation_event",
    "snow_event",
    "fog_or_low_visibility"
]

final_analysis_data = analysis_data.select(*final_columns)

FINAL_DATA_PATH = (PROCESSED_DIR / "airport_weather_trips_2024_h1")

final_analysis_data.coalesce(4).write.mode("overwrite").parquet(str(FINAL_DATA_PATH))

print(f"Final analysis trips saved: {analysis_trip_count:,}")

In [ ]:
weather_shape_check = [
    ("JFK raw weather", jfk_weather_raw),
    ("LaGuardia raw weather", lga_weather_raw),
    ("Selected weather observations", weather_selected),
    ("Hourly weather data", weather_hourly),
    ("Joined taxi-weather data", airport_weather_trips),
    ("Complete analysis data", analysis_data),
    ("Final selected data", final_analysis_data)
]

for name, dataframe in weather_shape_check:
    print(
        f"{name}: "
        f"{dataframe.count():,} rows x "
        f"{len(dataframe.columns)} columns"
    )